In [ ]:
%cd ../..
import os
import torch
import polars as pl
import numpy as np
from tqdm import tqdm
from omegaconf import OmegaConf

from dinov2.inference import generate_embeddings, build_model, view_volume, crop_volume

In [ ]:
data_path = "/scratch/scratch0/BIMCV-R/"

metadata_df = pl.read_csv(os.path.join(data_path, "curated_ct_report_path_En.csv"))
metadata_df = metadata_df.select(["path", "Report", "Report_en", "Labels"])
metadata_df.head()

In [ ]:
import nibabel as nib
from einops import rearrange

def load_nifti(path):
    nifti = nib.loadsave.load(path)
    image = nifti.dataobj[:]  # type: ignore
    affine = nifti.affine  # type: ignore

    s = np.sqrt((affine[:3, :3] ** 2).sum(axis=0))
    spacing = (float(s[2]), float(s[0]), float(s[1]))
    assert abs(spacing[2] - spacing[1]) < 0.001

    image = torch.from_numpy(image).float()
    image = rearrange(image, "w h d -> d w h")
    image = image.clip(-1000, 1900)

    return image, spacing

In [ ]:
sample_path = os.path.join(data_path, "nii/sub-S321876_ses-E44350_run-1_bp-chest_ct.nii")

img, spacing = load_nifti(sample_path)
view_volume(img)

In [ ]:
config_path = "/home/48078029W/projects/radio-foundation/runs/base10pat/config.yaml"
checkpoint_path = "/home/48078029W/projects/radio-foundation/runs/base10pat/eval/training_99999/teacher_checkpoint.pth"

device = torch.device("cuda")

config = OmegaConf.load(config_path)
model, autocast_ctx = build_model(checkpoint_path, config, img_size=504, device=device)

data_kwargs = dict(
    fmean = -573.8,
    fstd = 461.3,
    channels = 10,
    img_size = 504,
    patch_size = 14,
    device="cuda",
    block_size=64,
    autocast_ctx=autocast_ctx,
)

In [ ]:
output_path = "/scratch/scratch1/embeddings/RSNA-PE/demo"
os.makedirs(output_path, exist_ok=True)

for row in tqdm(metadata_df.iter_rows(), total=len(metadata_df)):
    mapid = row[0].replace(".nii.gz", "")
    img_path = os.path.join(data_path, f"nii/{mapid}.nii")

    img_output_dir = os.path.join(output_path, f"{mapid}.pth")

    if os.path.exists(img_output_dir):
        continue
    
    img, spacing = load_nifti(img_path)
    img = crop_volume(img)
    
    collated_features = generate_embeddings(
        img,
        model=model,
        **data_kwargs # type: ignore
    )

    output = {"cls": collated_features["cls"]}

    torch.save(output, img_output_dir)
